# 11. Factorial state spaces

![Eight binary states, a partial query, and the accuracy ceiling that follows from it](../images/11_factorial_state_spaces.svg)

A result reported as one number often hides several conditions. This notebook builds the address book that keeps those conditions separate: complete states, the queries that reveal only part of a state, and the accuracy ceiling that partial information imposes.

**Learning goals:** enumerate Cartesian products, separate factor subsets from their assignments, generate partial queries without duplicates, preserve exact probabilities, and compute a partial-information ceiling. Everything here uses synthetic binary factors, and the [lesson 11 lecture](../lectures/11_factorial_state_spaces.md) carries the derivations.

In [ ]:
from fractions import Fraction
from itertools import combinations, product
import numpy as np
import matplotlib.pyplot as plt

SEED = 11
rng = np.random.default_rng(SEED)
np.set_printoptions(linewidth=90)
print(f"NumPy {np.__version__}; seed={SEED}")

## 1. Cartesian products and shapes

Start with the complete states, since every later idea is a restriction of them.

For factor domains $\mathcal V_1,\ldots,\mathcal V_d$, the state space is $\Omega=\mathcal V_1\times\cdots\times\mathcal V_d$, and its size is the product of the domain sizes. The code writes the factor count as `d`, which the lecture calls $F$. A binary state is one row in $\{0,1\}^d$, so stacking all states gives an array of shape `(number of states, d)`. The assertions check both the count and that no state is duplicated.

In [ ]:
d = 3
states = np.array(list(product([0, 1], repeat=d)), dtype=np.int8)
expected_count = 2 ** d
assert states.shape == (expected_count, d)
assert np.unique(states, axis=0).shape[0] == expected_count
print("states shape:", states.shape)
print(states)

## 2. Bit-vector indexing

Each binary state is a string of bits, so it also names a single integer. Reading the bits as a binary number gives the index $\sum_j x_j2^{d-1-j}$, where the leftmost bit carries the largest place value.

Matrix multiplication against the vector of place values computes every index at once, which is both clearer and faster than a Python loop over a materialized table. The assertion confirms that lexicographic enumeration and integer order agree. For very many binary rows, `np.packbits` stores the same information in a fraction of the space.

In [ ]:
bit_weights = 2 ** np.arange(d - 1, -1, -1)
state_indices = states @ bit_weights
assert np.array_equal(state_indices, np.arange(2 ** d))
packed = np.packbits(states, axis=1, bitorder="big")
print("weights:", bit_weights, "indices:", state_indices.tolist())
print("packed storage shape:", packed.shape)

## 3. Factor subsets, complements, and queries

A query usually reveals only part of a state, so we need to name the revealed part. A subset $S$ names the known coordinates, and its complement $S^c$ holds the rest relative to the full index set.

A subset alone is not a query: a query is the pair `(S, values)`. Counting them makes the distinction concrete. Every coordinate has three query statuses, omitted or fixed to 0 or fixed to 1, so there are $3^d$ partial queries in total, including the empty query that reveals nothing.

In [ ]:
def all_subsets(num_factors):
    for size in range(num_factors + 1):
        yield from combinations(range(num_factors), size)

def binary_queries(num_factors):
    for subset in all_subsets(num_factors):
        for assignment in product([0, 1], repeat=len(subset)):
            yield subset, assignment

queries = list(binary_queries(d))
assert len(queries) == 3 ** d
assert len(set(queries)) == len(queries)
S = (0, 2)
Sc = tuple(j for j in range(d) if j not in S)
assert Sc == (1,)
print(f"{len(queries)} unique queries; S={S}, complement={Sc}")

## 4. Exact partial-information ceilings

Now use those queries to bound accuracy. A query revealing factors 0 and 2 leaves factor 1 unknown, so under a uniform prior exactly two complete states match each observed assignment, and the Bayes-optimal success probability is exactly $1/2$.

These quantities are ratios of small integers, so nothing here needs floating point. `fractions.Fraction` keeps the result exact and lets the assertion compare against `Fraction(1, 2)` rather than a tolerance. Fractions are ideal for small enumerations and slower than vectorized floating-point arrays for large ones.

In [ ]:
def compatible_rows(table, subset, assignment):
    subset = np.asarray(subset, dtype=int)
    assignment = np.asarray(assignment, dtype=table.dtype)
    return np.all(table[:, subset] == assignment, axis=1)

assignments = list(product([0, 1], repeat=len(S)))
group_sizes = [int(compatible_rows(states, S, a).sum()) for a in assignments]
ceiling = sum((Fraction(size, len(states)) * Fraction(1, size)
               for size in group_sizes), start=Fraction(0, 1))
assert group_sizes == [2, 2, 2, 2]
assert ceiling == Fraction(1, 2)
print("compatible states per query:", group_sizes)
print("exact average ceiling:", ceiling, "=", float(ceiling))

## 5. Visualize one query

The compatibility test above is worth seeing as a picture. The Boolean mask is computed across every state at once, and `np.all(..., axis=1)` reduces the per-coordinate comparisons to one flag per row.

The bar chart shows all eight complete states and highlights the two compatible with the query `1 ? 1`, where `?` marks the unrevealed factor. Those two states are exactly the ones the final assertion names.

In [ ]:
mask = compatible_rows(states, S, (1, 1))
colors = np.where(mask, "#d89b21", "#c9d8e6")
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.bar(state_indices, np.ones(len(states)), color=colors, edgecolor="#52606d")
ax.set(xticks=state_indices, xticklabels=["".join(map(str, row)) for row in states],
       yticks=[], xlabel="complete state", title="States compatible with query 1 ? 1")
ax.set_ylim(0, 1.25)
plt.tight_layout()
plt.show()
assert state_indices[mask].tolist() == [5, 7]

## 6. GFC-v2 complementary donors and exact oracles

The same machinery now builds the study's real queries. Note first which factorial design this is: a factorial outcome state space crossing speed, clothing, and direction within one participant, not the factorial training experiment that crosses sequence support with temporal policy.

We enumerate all eight outcome cells and both allowed focal factors, giving 16 queries. Every composed query draws its blocks from donors only, and the target contributes the answer alone. Before construction, each query checks donor lineage against the target and keeps all eight recordings in the gallery.

The synthetic source IDs are chosen to make that lineage check meaningful: opposite directions at fixed speed and clothing share one source ID, so the assertion tests physical source separation rather than mere cell inequality. That is also why direction-focal queries must fail, and the loop counts all eight rejections.

In [ ]:
factor_names = ("speed", "clothing", "direction")
gfc_cells = [tuple(int(value) for value in row) for row in states]
recording_id = {cell: f"recording-{cell}" for cell in gfc_cells}
source_video_id = {cell: f"source-speed{cell[0]}-clothing{cell[1]}"
                   for cell in gfc_cells}
gallery_ids = tuple(recording_id[cell] for cell in gfc_cells)

def gfc_donors(target, focal):
    focal_index = factor_names.index(focal)
    donor_u = tuple(value if j == focal_index else 1 - value
                    for j, value in enumerate(target))
    donor_v = tuple(1 - value if j == focal_index else value
                    for j, value in enumerate(target))
    return donor_u, donor_v

def require_source_separation(target, donor_u, donor_v):
    target_source = source_video_id[target]
    if source_video_id[donor_u] == target_source or source_video_id[donor_v] == target_source:
        raise ValueError('a donor shares source_video_id with the target')

gfc_queries = []
for target in gfc_cells:
    for focal in ("speed", "clothing"):
        donor_u, donor_v = gfc_donors(target, focal)
        require_source_separation(target, donor_u, donor_v)
        factor_sources = tuple("donor_u" if name == focal else "donor_v"
                               for name in factor_names)
        query = {"target": recording_id[target],
                 "target_cell": target,
                 "focal": focal,
                 "donor_u": recording_id[donor_u],
                 "donor_v": recording_id[donor_v],
                 "target_source_video_id": source_video_id[target],
                 "donor_source_video_ids": (source_video_id[donor_u], source_video_id[donor_v]),
                 "factor_sources": factor_sources,
                 "gallery_ids": gallery_ids}
        assert all(source in {"donor_u", "donor_v"}
                   for source in factor_sources)  # no target features
        assert query["donor_u"] in gallery_ids and query["donor_v"] in gallery_ids
        gfc_queries.append(query)

direction_focal_rejections = 0
for target in gfc_cells:
    donor_u, donor_v = gfc_donors(target, 'direction')
    try:
        require_source_separation(target, donor_u, donor_v)
    except ValueError:
        direction_focal_rejections += 1
    else:
        raise AssertionError('direction-focal query must fail source separation')

information_oracles = {k: Fraction(1, 2 ** (3 - k)) for k in range(4)}
assert len(gfc_queries) == 16
assert len({(q["target"], q["donor_u"], q["donor_v"]) for q in gfc_queries}) == 16
assert direction_focal_rejections == 8
assert information_oracles == {0: Fraction(1, 8), 1: Fraction(1, 4),
                               2: Fraction(1, 2), 3: Fraction(1, 1)}
print("GFC-v2 query count:", len(gfc_queries))
print("zero-to-three-factor oracles:", information_oracles)

## Exercises and takeaways

1. Change `d` to 4. Predict the numbers of complete states and partial queries before running.
2. Reveal only factor 0. Compute the ceiling. Then reveal no factors.
3. Replace one binary domain with three values and verify that state counts multiply.

**Brief answers:** 4 binary factors give 16 states and 81 queries. Revealing one of three binary factors leaves 4 compatible states, so the ceiling is $1/4$; revealing none gives $1/8$. A ternary factor multiplies the corresponding state count by 3 rather than 2.

**Takeaway:** factorial spaces separate complete states from partial observations. The ambiguity a query leaves behind sets a ceiling that no classifier can exceed without additional information, so the ceiling describes the query rather than the model.

## Continue learning

Lesson 12 turns these cells into gallery rows and scores them with blockwise distances.

[Previous notebook: 10](10_regularized_linear_estimation.ipynb) | [Lecture](../lectures/11_factorial_state_spaces.md) | [Curriculum](../README.md) | [Next notebook: 12](12_blockwise_distances_and_ranking.ipynb)